# Gold Medal Tabular Modeling Baseline

競馬予測への金メダル解法戦略の適用。

16コンペ（Optiver, AMEX, IEEE Fraud, Home Credit, Ubiquant, Child Mind, ICR, Equity HCT, Enefit, Mitsui 等）の
金メダル解法から抽出した共通パターンを keiba-vpn に実装する。

## 実装する戦略

| 戦略 | 参考コンペ |
|---|---|
| LGBM + XGB + CatBoost 3モデルアンサンブル | 全コンペ共通 |
| ラグ特徴・ローリング統計 | Optiver, Enefit, Ubiquant |
| グループ集約（同日同開催コンテキスト） | Ubiquant time_id 集約 |
| 2段階分解（分類 + ランキング） | Equity HCT |
| Purged GroupKFold（レース日付でグループ化） | IEEE Fraud, Jane Street |
| 多シード平均（ノイズ安定化） | Child Mind, ICR |
| Adversarial Validation（分布シフト検出） | IEEE Fraud, AMEX |
| 予測後処理（レース内ランク整合） | Optiver zero-sum, IEEE Fraud UID averaging |


## 0. Setup

In [1]:
from __future__ import annotations

import sys
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", 80)
pd.set_option("display.float_format", "{:.4f}".format)

# プロジェクトルート解決
_root = Path.cwd().resolve()
for _ in range(16):
    if (_root / "requirements.txt").is_file() and (_root / ".env.example").is_file():
        break
    _root = _root.parent
else:
    raise RuntimeError("keiba-vpn ルートが見つかりません")
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))

REPO_ROOT = _root
print("REPO_ROOT:", REPO_ROOT)

REPO_ROOT: /home/hirokiakataoka/project/myproject/project-multi-agent/project/keiba-vpn/repo


In [2]:
# ライブラリ確認
import importlib

REQUIRED = ["lightgbm", "xgboost", "catboost", "sklearn", "optuna"]
OPTIONAL = ["shap"]

for lib in REQUIRED:
    try:
        m = importlib.import_module(lib)
        print(f"  OK  {lib} {getattr(m, '__version__', '')}")
    except ImportError:
        print(f"  NG  {lib} — pip install {lib}")

for lib in OPTIONAL:
    try:
        m = importlib.import_module(lib)
        print(f"  OK  {lib} {getattr(m, '__version__', '')} (optional)")
    except ImportError:
        print(f"  --  {lib} not installed (optional — SHAP 解釈は skip)")

  OK  lightgbm 4.6.0
  OK  xgboost 3.2.0
  OK  catboost 1.2.10
  OK  sklearn 1.8.0
  OK  optuna 4.8.0
  --  shap not installed (optional — SHAP 解釈は skip)


## 1. モックデータ生成（全データソース対応）

実データなしでパイプライン全体を検証するためのモック DataFrame を生成する。  
`make_mock_df()` は以下の全データソースを統合した構成:

| ソース | 代表カラム |
|---|---|
| race_shutuba（レース情報） | race_date, venue_code, surface, distance, grade, field_size, … |
| race_shutuba（出馬表） | horse_id, jockey_id, weight, sire, dam_sire, … |
| race_index（速度指数） | idx_speed_max/avg/distance/course, idx_speed_recent_1〜3 |
| historical_entity_stats | horse/jockey/trainer_prior_races, _win_rate, _top3_rate, … |
| track_bias_pedigree | track_bias_winner_3f_weighted, sire_x_track_bias_weighted, … |
| jockey_trainer_stats | jockey/trainer_roll5/10_*, cal90/365_starts/wins |
| label（race_result） | finish_position |

実データがある場合は Cell 5 のフラグを `USE_MOCK = False` に切り替える。

In [3]:
USE_MOCK = True  # False にすると LAYER_A_CACHE / build_layer_a_dataframe() を使用


def make_mock_df(n_races: int = 250, max_horses: int = 16, seed: int = 42) -> pd.DataFrame:
    """
    全データソース対応のモック DataFrame を生成する。
    race_shutuba / race_index / entity_stats / track_bias_pedigree / jt_stats を統合。
    """
    rng = np.random.default_rng(seed)

    venues = ["tokyo", "nakayama", "kyoto", "hanshin", "chukyo", "kokura",
              "niigata", "fukushima", "sapporo", "hakodate"]
    surfaces = ["turf", "dirt"]
    grades = ["G1", "G2", "G3", "OP", "3勝", "2勝", "1勝", "未勝利", "新馬"]
    conditions = ["firm", "good", "yielding", "soft"]
    weathers = ["sunny", "cloudy", "rainy"]
    sires = ["Deep Impact", "Heart's Cry", "Daiwa Major", "Stay Gold",
             "King Kamehameha", "Orfevre", "Duramente", "Epiphaneia"]
    dam_sires = ["Sunday Silence", "Brian's Time", "Seeking the Gold",
                 "Northern Taste", "Tony Bin", "Hector Protector"]

    horse_ids = [f"H{i:05d}" for i in range(400)]
    jockey_ids = [f"J{i:04d}" for i in range(60)]
    trainer_ids = [f"T{i:04d}" for i in range(70)]

    rows = []
    base_date = pd.Timestamp("2022-01-08")
    distances = [1000, 1200, 1400, 1600, 1800, 2000, 2200, 2400, 3000, 3200]

    for race_idx in range(n_races):
        date = base_date + pd.Timedelta(days=(race_idx // 8) * 7 + race_idx % 2)
        venue_code = rng.choice(venues)
        surface = rng.choice(surfaces)
        distance = int(rng.choice(distances))
        grade = rng.choice(grades)
        n_horses = int(rng.integers(8, max_horses + 1))
        race_id = f"{date.strftime('%Y%m%d')}{venue_code[:2].upper()}{race_idx % 12 + 1:02d}"

        selected_horses = rng.choice(horse_ids, size=n_horses, replace=False)
        finish_order = rng.permutation(n_horses) + 1
        # 人気順: 馬の事前勝率 + ノイズで決定し、順位付け
        popularity_scores = rng.standard_normal(n_horses)
        popularity_order = n_horses - popularity_scores.argsort().argsort()  # 1=最高評価

        for h_num, (horse_id, finish_pos, pop) in enumerate(zip(selected_horses, finish_order, popularity_order), 1):
            jockey_id = rng.choice(jockey_ids)
            trainer_id = rng.choice(trainer_ids)
            h_prior = int(rng.integers(0, 50))
            h_wins = int(rng.integers(0, max(1, h_prior // 5 + 1)))
            h_top3 = int(rng.integers(h_wins, max(h_wins + 1, h_prior // 3 + 1)))

            # 単勝モーニングオッズ: 人気が高い(pop=1)ほど低オッズ
            morning_odds = round(float(pop) * rng.uniform(0.8, 1.5) + rng.uniform(0, 2), 1)
            morning_odds = max(1.0, morning_odds)

            row = {
                # ── 主キー ──
                "race_id": race_id,
                "horse_number": h_num,
                "horse_id": horse_id,

                # ── race_shutuba: レース情報 ──
                "race_date": date,
                "venue": venue_code,
                "venue_code": venue_code,
                "surface": surface,
                "distance": float(distance),
                "direction": rng.choice(["right", "left"]),
                "grade": grade,
                "race_class": grade,
                "weather": rng.choice(weathers),
                "track_condition": rng.choice(conditions),
                "start_time": f"{int(rng.integers(10,17)):02d}:{int(rng.choice([0,5,10,15,20,25,30,35,40,45,50,55])):02d}",
                "field_size": float(n_horses),
                "race_name": f"MockRace{race_idx:03d}",
                "round": float(race_idx % 8 + 1),
                "weight_rule": rng.choice(["handicap", "weight-for-age", "conditions"]),
                "course_type": rng.choice(["inner", "outer", "straight"]),

                # ── race_shutuba: 出馬表 ──
                "bracket_number": float((h_num - 1) // 2 + 1),
                "horse_name": f"MockHorse{horse_id}",
                "sex_age": f"{rng.choice(['牡','牝','セ'])}{int(rng.integers(2,8))}",
                "jockey_weight": float(rng.integers(50, 60)),
                "jockey_name": f"MockJockey{jockey_id}",
                "jockey_id": jockey_id,
                "trainer_name": f"MockTrainer{trainer_id}",
                "trainer_id": trainer_id,
                "weight": float(rng.integers(420, 560)),
                "weight_change": float(rng.integers(-8, 9)),
                "sire": rng.choice(sires),
                "dam_sire": rng.choice(dam_sires),

                # ── 市場情報（レース前に確定する特徴量）──
                "popularity": float(pop),           # 人気順位（1=1番人気）
                "morning_odds": morning_odds,        # モーニングオッズ（単勝）

                # ── race_index: 速度指数 ──
                "idx_speed_max": float(rng.integers(70, 120)),
                "idx_speed_avg": float(rng.integers(65, 115)),
                "idx_speed_distance": float(rng.integers(60, 110)),
                "idx_speed_course": float(rng.integers(60, 110)),
                "idx_speed_recent_1": float(rng.integers(60, 115)),
                "idx_speed_recent_2": float(rng.integers(55, 115)) if rng.random() > 0.2 else np.nan,
                "idx_speed_recent_3": float(rng.integers(55, 115)) if rng.random() > 0.3 else np.nan,

                # ── historical_entity_stats: 馬 ──
                "horse_prior_races": float(h_prior),
                "horse_prior_wins": float(h_wins),
                "horse_prior_top3": float(h_top3),
                "horse_prior_win_rate": float(h_wins / max(1, h_prior)),
                "horse_prior_top3_rate": float(h_top3 / max(1, h_prior)),
                "horse_prior_avg_finish": float(rng.uniform(2.0, 12.0)),

                # ── historical_entity_stats: 騎手 ──
                "jockey_prior_races": float(rng.integers(100, 2000)),
                "jockey_prior_wins": float(rng.integers(10, 300)),
                "jockey_prior_top3": float(rng.integers(30, 600)),
                "jockey_prior_win_rate": float(rng.uniform(0.05, 0.30)),
                "jockey_prior_top3_rate": float(rng.uniform(0.15, 0.50)),

                # ── historical_entity_stats: 調教師 ──
                "trainer_prior_races": float(rng.integers(50, 1500)),
                "trainer_prior_wins": float(rng.integers(5, 200)),
                "trainer_prior_top3": float(rng.integers(15, 500)),
                "trainer_prior_win_rate": float(rng.uniform(0.05, 0.25)),
                "trainer_prior_top3_rate": float(rng.uniform(0.10, 0.40)),

                # ── track_bias_pedigree ──
                "track_bias_winner_3f_same_day_prior": float(rng.uniform(32, 38)) if rng.random() > 0.3 else np.nan,
                "track_bias_winner_3f_prev_day": float(rng.uniform(32, 38)) if rng.random() > 0.2 else np.nan,
                "track_bias_winner_3f_diff": float(rng.uniform(-1, 1)) if rng.random() > 0.3 else np.nan,
                "track_bias_winner_3f_weighted": float(rng.uniform(32, 38)) if rng.random() > 0.15 else np.nan,
                "sire_x_track_bias_weighted": float(rng.uniform(0, 5)) if rng.random() > 0.2 else np.nan,
                "dam_sire_x_track_bias_weighted": float(rng.uniform(0, 5)) if rng.random() > 0.2 else np.nan,
                "sire_x_track_bias_diff": float(rng.uniform(-1, 1)) if rng.random() > 0.3 else np.nan,
                "dam_sire_x_track_bias_diff": float(rng.uniform(-1, 1)) if rng.random() > 0.3 else np.nan,

                # ── jockey_trainer_stats (jt_race_features) ──
                "jockey_roll5_starts": float(rng.integers(3, 6)),
                "jockey_roll5_wins": float(rng.integers(0, 3)),
                "jockey_roll5_top3": float(rng.integers(0, 4)),
                "jockey_roll5_avg_finish": float(rng.uniform(2, 10)),
                "jockey_roll10_starts": float(rng.integers(5, 11)),
                "jockey_roll10_wins": float(rng.integers(0, 5)),
                "jockey_roll10_top3": float(rng.integers(0, 7)),
                "jockey_cal90_starts": float(rng.integers(10, 80)),
                "jockey_cal90_wins": float(rng.integers(1, 20)),
                "jockey_cal365_starts": float(rng.integers(30, 300)),
                "jockey_cal365_wins": float(rng.integers(3, 80)),
                "trainer_roll5_starts": float(rng.integers(2, 6)),
                "trainer_roll5_wins": float(rng.integers(0, 3)),
                "trainer_roll5_top3": float(rng.integers(0, 4)),
                "trainer_roll5_avg_finish": float(rng.uniform(2, 10)),
                "trainer_cal90_starts": float(rng.integers(5, 50)),
                "trainer_cal90_wins": float(rng.integers(0, 15)),
                "trainer_cal365_starts": float(rng.integers(20, 200)),
                "trainer_cal365_wins": float(rng.integers(2, 60)),

                # ── label (race_result) ──
                "finish_position": float(finish_pos),
            }
            rows.append(row)

    df = pd.DataFrame(rows)
    df["race_date"] = pd.to_datetime(df["race_date"])
    return df


# ── データ取得 ──
if USE_MOCK:
    print("モックデータを生成...")
    df_raw = make_mock_df(n_races=250, max_horses=16, seed=42)
    print(f"モック完了: {df_raw.shape}")
else:
    LAYER_A_CACHE = REPO_ROOT / "data" / "local" / "modeling" / "layer_a_train.parquet"
    if LAYER_A_CACHE.exists():
        print("キャッシュから読み込み:", LAYER_A_CACHE)
        df_raw = pd.read_parquet(LAYER_A_CACHE)
    else:
        print("キャッシュなし → build_layer_a_dataframe() で構築")
        from src.pipeline.models.layer_a_dataset import build_layer_a_dataframe
        df_raw = build_layer_a_dataframe(base_dir=REPO_ROOT)

print(f"Shape: {df_raw.shape}")
print(f"Columns ({len(df_raw.columns)}): {list(df_raw.columns[:25])} ...")
df_raw.head(3)

モックデータを生成...


モック完了: (2999, 84)
Shape: (2999, 84)
Columns (84): ['race_id', 'horse_number', 'horse_id', 'race_date', 'venue', 'venue_code', 'surface', 'distance', 'direction', 'grade', 'race_class', 'weather', 'track_condition', 'start_time', 'field_size', 'race_name', 'round', 'weight_rule', 'course_type', 'bracket_number', 'horse_name', 'sex_age', 'jockey_weight', 'jockey_name', 'jockey_id'] ...


,race_id,horse_number,horse_id,race_date,venue,venue_code,surface,distance,direction,grade,race_class,weather,track_condition,start_time,field_size,race_name,round,weight_rule,course_type,bracket_number,horse_name,sex_age,jockey_weight,jockey_name,jockey_id,trainer_name,trainer_id,weight,weight_change,sire,dam_sire,popularity,morning_odds,idx_speed_max,idx_speed_avg,idx_speed_distance,idx_speed_course,idx_speed_recent_1,idx_speed_recent_2,idx_speed_recent_3,...,horse_prior_top3_rate,horse_prior_avg_finish,jockey_prior_races,jockey_prior_wins,jockey_prior_top3,jockey_prior_win_rate,jockey_prior_top3_rate,trainer_prior_races,trainer_prior_wins,trainer_prior_top3,trainer_prior_win_rate,trainer_prior_top3_rate,track_bias_winner_3f_same_day_prior,track_bias_winner_3f_prev_day,track_bias_winner_3f_diff,track_bias_winner_3f_weighted,sire_x_track_bias_weighted,dam_sire_x_track_bias_weighted,sire_x_track_bias_diff,dam_sire_x_track_bias_diff,jockey_roll5_starts,jockey_roll5_wins,jockey_roll5_top3,jockey_roll5_avg_finish,jockey_roll10_starts,jockey_roll10_wins,jockey_roll10_top3,jockey_cal90_starts,jockey_cal90_wins,jockey_cal365_starts,jockey_cal365_wins,trainer_roll5_starts,trainer_roll5_wins,trainer_roll5_top3,trainer_roll5_avg_finish,trainer_cal90_starts,trainer_cal90_wins,trainer_cal365_starts,trainer_cal365_wins,finish_position
0,20220108TO01,1,H00037,2022-01-08,tokyo,tokyo,dirt,2200.0000,right,OP,OP,rainy,good,12:10,11.0000,MockRace000,1.0000,weight-for-age,straight,1.0000,MockHorseH00037,セ4,51.0000,MockJockeyJ0024,J0024,MockTrainerT0022,T0022,536.0000,2.0000,Orfevre,Sunday Silence,11.0000,10.5000,85.0000,103.0000,101.0000,81.0000,104.0000,108.0000,72.0000,...,0.0667,3.3975,1682.0000,67.0000,488.0000,0.2467,0.3827,60.0000,96.0000,357.0000,0.2061,0.2377,32.8388,NaN,-0.0578,36.5900,2.7679,1.5198,NaN,-0.5708,3.0000,1.0000,3.0000,3.8715,10.0000,4.0000,0.0000,69.0000,6.0000,277.0000,25.0000,3.0000,1.0000,0.0000,8.2712,30.0000,14.0000,139.0000,25.0000,11.0000
1,20220108TO01,2,H00286,2022-01-08,tokyo,tokyo,dirt,2200.0000,left,OP,OP,rainy,yielding,13:40,11.0000,MockRace000,1.0000,handicap,straight,1.0000,MockHorseH00286,牝7,51.0000,MockJockeyJ0024,J0024,MockTrainerT0029,T0029,489.0000,3.0000,Stay Gold,Seeking the Gold,2.0000,1.8000,78.0000,84.0000,71.0000,75.0000,97.0000,92.0000,NaN,...,0.1000,3.1801,743.0000,288.0000,238.0000,0.2249,0.2431,1367.0000,154.0000,485.0000,0.2058,0.3151,33.6334,NaN,-0.0884,33.8357,0.8839,3.7926,-0.1358,0.1682,3.0000,1.0000,2.0000,5.3265,5.0000,3.0000,0.0000,22.0000,10.0000,80.0000,28.0000,4.0000,0.0000,2.0000,6.7012,9.0000,11.0000,50.0000,19.0000,9.0000
2,20220108TO01,3,H00314,2022-01-08,tokyo,tokyo,dirt,2200.0000,right,OP,OP,rainy,good,13:25,11.0000,MockRace000,1.0000,conditions,inner,2.0000,MockHorseH00314,牡3,54.0000,MockJockeyJ0055,J0055,MockTrainerT0069,T0069,513.0000,0.0000,Stay Gold,Hector Protector,7.0000,8.5000,78.0000,93.0000,87.0000,83.0000,104.0000,71.0000,105.0000,...,0.1379,2.2161,933.0000,115.0000,500.0000,0.2740,0.1991,488.0000,113.0000,168.0000,0.1844,0.1844,36.3620,32.6464,-0.5396,NaN,1.8546,4.0413,0.9058,NaN,3.0000,1.0000,2.0000,4.0477,10.0000,4.0000,1.0000,21.0000,11.0000,42.0000,26.0000,3.0000,2.0000,3.0000,9.1334,9.0000,11.0000,172.0000,53.0000,2.0000


In [4]:
# 日付列の確認と型変換
DATE_COLS = [c for c in ["race_date", "kaisai_date", "date"] if c in df_raw.columns]
DATE_COL = DATE_COLS[0] if DATE_COLS else None
print("日付列:", DATE_COL)

if DATE_COL:
    df_raw[DATE_COL] = pd.to_datetime(df_raw[DATE_COL], errors="coerce")
    print(f"期間: {df_raw[DATE_COL].min()} 〜 {df_raw[DATE_COL].max()}")
    print(f"レース数: {df_raw['race_id'].nunique():,}")
    print(f"馬数（延べ）: {len(df_raw):,}")

# ターゲット確認
TARGET_COLS = [c for c in ["finish_position", "rank", "finish_pos"] if c in df_raw.columns]
TARGET_COL = TARGET_COLS[0] if TARGET_COLS else None
print("\nターゲット列:", TARGET_COL)
if TARGET_COL:
    print(df_raw[TARGET_COL].value_counts().sort_index().head(20))

日付列: race_date
期間: 2022-01-08 00:00:00 〜 2022-08-14 00:00:00
レース数: 250
馬数（延べ）: 2,999

ターゲット列: finish_position
finish_position
1.0000     250
2.0000     250
3.0000     250
4.0000     250
5.0000     250
6.0000     250
7.0000     250
8.0000     250
9.0000     225
10.0000    194
11.0000    163
12.0000    133
13.0000    112
14.0000     84
15.0000     58
16.0000     30
Name: count, dtype: int64


## 2. 特徴量エンジニアリング（金メダル戦略）

### 2-1. ラグ特徴・ローリング統計

> Optiver 1位: "Feature engineering was the primary driver of performance."  
> Enefit: 2〜14日前のラグが全解法で共通  
> Home Credit: "time-sliced windows rather than static aggregates"

In [5]:
def add_horse_sequential_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    馬単位の時系列特徴量を追加する。
    - 前走タイム・着順ラグ (lag 1〜3)
    - 直近 3/5 走ローリング統計（平均・標準偏差）
    - 前走からの日数差
    
    ※ leakage 防止: sort → shift のみ使用（当日情報は含めない）
    """
    df = df.copy()
    df = df.sort_values(["horse_id", date_col]).reset_index(drop=True)

    NUMERIC_COLS = [c for c in [
        "finish_time_sec", "last_3f_sec", "speed_max", "speed_avg",
        "weight_carried", "horse_weight", "speed_recent",
    ] if c in df.columns]

    target_like = TARGET_COL  # 'finish_position' 等

    grp = df.groupby("horse_id", sort=False)

    for col in NUMERIC_COLS + ([target_like] if target_like else []):
        for lag in [1, 2, 3]:
            df[f"{col}_lag{lag}"] = grp[col].shift(lag)

    for col in NUMERIC_COLS + ([target_like] if target_like else []):
        for window in [3, 5]:
            rolled = grp[col].transform(lambda s: s.shift(1).rolling(window, min_periods=1).mean())
            df[f"{col}_roll{window}mean"] = rolled
            rolled_std = grp[col].transform(lambda s: s.shift(1).rolling(window, min_periods=2).std())
            df[f"{col}_roll{window}std"] = rolled_std

    if date_col in df.columns:
        days_since = grp[date_col].transform(lambda s: (s - s.shift(1)).dt.days)
        df["days_since_prev_race"] = days_since

    return df


print("add_horse_sequential_features() 定義完了")

add_horse_sequential_features() 定義完了


### 2-2. グループ集約特徴（同日同開催コンテキスト）

> Ubiquant 1位: "time_id aggregation — CV 0.141 → 0.154"  
> IEEE Fraud: "frequency encoding on high-cardinality features"

競馬では `race_date × venue_code` が Ubiquant の `time_id` に相当。  
同じ馬場・同じ日の他レースのオッズや速度指数は強い文脈特徴になる。

In [6]:
def add_race_context_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    同日同開催（race_date × venue_code）内の集約特徴を追加する。
    Ubiquant の time_id 集約に相当。
    """
    df = df.copy()

    venue_col = next((c for c in ["venue_code", "course_code", "venue"] if c in df.columns), None)
    group_key = [date_col, venue_col] if venue_col else [date_col]

    AGGREGATE_COLS = [c for c in [
        "speed_max", "speed_avg", "weight_carried",
    ] if c in df.columns]

    for col in AGGREGATE_COLS:
        ctx_mean = df.groupby(group_key)[col].transform("mean")
        ctx_std = df.groupby(group_key)[col].transform("std")
        df[f"{col}_ctx_mean"] = ctx_mean
        df[f"{col}_ctx_std"] = ctx_std
        # 同コンテキスト内での偏差
        df[f"{col}_ctx_dev"] = df[col] - ctx_mean

    # レース内出走頭数
    df["n_horses_in_race"] = df.groupby("race_id")["race_id"].transform("count")

    # 騎手・調教師の頻度エンコーディング（high cardinality 対策）
    for entity_col in ["jockey_id", "trainer_id"]:
        if entity_col in df.columns:
            freq = df[entity_col].map(df[entity_col].value_counts())
            df[f"{entity_col}_freq"] = freq

    return df


print("add_race_context_features() 定義完了")

add_race_context_features() 定義完了


### 2-3. ドメイン特有の特徴量

> Home Credit: "Domain-Specific Metrics — Interest Rate, Credit Utilization"  
> Enefit: `installed_capacity * solar_radiation / (temperature + 273.15)`

In [7]:
def add_domain_features(df: pd.DataFrame) -> pd.DataFrame:
    """
    競馬ドメイン固有の特徴量を追加する。
    - 斤量負担率（weight_carried / horse_weight）
    - 前走着差の回復指標
    - 直近ラグ間のトレンド（速度指数の傾き）
    """
    df = df.copy()

    # 斤量負担率（軽い馬ほど斤量の影響が大きい）
    if "weight_carried" in df.columns and "horse_weight" in df.columns:
        hw = pd.to_numeric(df["horse_weight"], errors="coerce")
        wc = pd.to_numeric(df["weight_carried"], errors="coerce")
        df["burden_ratio"] = wc / hw.replace(0, np.nan)

    # 速度指数トレンド（lag1 - lag2: 上昇中か下降中か）
    if "speed_max_lag1" in df.columns and "speed_max_lag2" in df.columns:
        df["speed_trend_1v2"] = df["speed_max_lag1"] - df["speed_max_lag2"]
    if "speed_max_lag2" in df.columns and "speed_max_lag3" in df.columns:
        df["speed_trend_2v3"] = df["speed_max_lag2"] - df["speed_max_lag3"]

    # 着順ラグ間のトレンド
    if TARGET_COL and f"{TARGET_COL}_lag1" in df.columns and f"{TARGET_COL}_lag2" in df.columns:
        df["rank_trend_1v2"] = df[f"{TARGET_COL}_lag2"] - df[f"{TARGET_COL}_lag1"]  # 正 = 改善

    return df


print("add_domain_features() 定義完了")

add_domain_features() 定義完了


### 2-4. 競馬固有追加特徴量（人気・季節性・血統適性）

> Home Credit: "Domain-Specific Metrics"  
> 競馬固有の事前情報（市場オッズ・季節性・血統 × 馬場適性）を特徴化する。

In [8]:
def add_keiba_specific_features(df: pd.DataFrame, date_col: str) -> pd.DataFrame:
    """
    競馬固有の追加特徴量:
    - 季節性（月・四季・夏競馬フラグ）
    - 人気 × 速度指数の乖離（市場の過小/過大評価検出）
    - 血統 × 馬場適性スコア（sire-surface / sire-距離帯）
    - オッズの対数変換・人気帯
    """
    df = df.copy()

    # ── 季節性 ──
    if date_col in df.columns:
        m = df[date_col].dt.month
        df["race_month"] = m.astype(float)
        df["race_season"] = ((m - 1) // 3).astype(float)   # 0=冬 1=春 2=夏 3=秋
        df["is_summer_racing"] = m.between(7, 9).astype(float)
        df["is_winter_racing"] = ((m == 12) | (m <= 2)).astype(float)

    # ── 人気・オッズ関連 ──
    if "popularity" in df.columns:
        df["popularity_inv"] = 1.0 / df["popularity"].clip(1)  # 1番人気=1.0
        df["popularity_tier"] = pd.cut(
            df["popularity"], bins=[0, 3, 7, 999], labels=[0, 1, 2]
        ).astype(float)  # 0=上位人気 1=中位 2=下位

    if "morning_odds" in df.columns:
        df["log_morning_odds"] = np.log1p(df["morning_odds"])
        # 隠れ確率（インプライド確率）= 1 / (1 + odds)
        df["implied_prob"] = 1.0 / (1.0 + df["morning_odds"].clip(0.1))

    # ── 人気と実力指数の乖離（市場非効率スコア）──
    if "popularity" in df.columns and "horse_prior_win_rate" in df.columns:
        # 人気が低い（数値大）のに勝率実績が高い = 過小評価馬
        df["market_inefficiency"] = (
            df["horse_prior_win_rate"] - df["popularity_inv"]
        )
        # レース内での速度指数順位 vs 人気順位の乖離
        if "idx_speed_max" in df.columns:
            speed_rank = df.groupby("race_id")["idx_speed_max"].rank(ascending=False)
            df["speed_rank_vs_popularity"] = df["popularity"] - speed_rank

    # ── 血統 × 馬場適性（同一 sire の surface 別 prior_win_rate 平均）──
    if "sire" in df.columns and "surface" in df.columns:
        df["sire_surface_affinity"] = df.groupby(["sire", "surface"])["horse_prior_win_rate"].transform("mean")

    # ── 血統 × 距離帯適性 ──
    if "sire" in df.columns and "distance" in df.columns:
        df["distance_band"] = pd.cut(
            df["distance"],
            bins=[0, 1400, 1800, 2200, 9999],
            labels=[0, 1, 2, 3],
        ).astype(float)
        df["sire_distance_affinity"] = df.groupby(["sire", "distance_band"])["horse_prior_win_rate"].transform("mean")

    return df


print("add_keiba_specific_features() 定義完了")

add_keiba_specific_features() 定義完了


In [9]:
# 特徴量パイプライン実行
if DATE_COL and "horse_id" in df_raw.columns:
    df_fe = add_horse_sequential_features(df_raw, DATE_COL)
    df_fe = add_race_context_features(df_fe, DATE_COL)
    df_fe = add_domain_features(df_fe)
    df_fe = add_keiba_specific_features(df_fe, DATE_COL)
    print(f"特徴量追加後 shape: {df_fe.shape}")
    new_cols = [c for c in df_fe.columns if c not in df_raw.columns]
    print(f"新規特徴量数: {len(new_cols)}")
    print("サンプル:", new_cols[:20])
else:
    df_fe = df_raw.copy()
    print("WARNING: horse_id または date_col が見つからないため特徴量追加をスキップ")

特徴量追加後 shape: (2999, 109)
新規特徴量数: 25
サンプル: ['finish_position_lag1', 'finish_position_lag2', 'finish_position_lag3', 'finish_position_roll3mean', 'finish_position_roll3std', 'finish_position_roll5mean', 'finish_position_roll5std', 'days_since_prev_race', 'n_horses_in_race', 'jockey_id_freq', 'trainer_id_freq', 'rank_trend_1v2', 'race_month', 'race_season', 'is_summer_racing', 'is_winter_racing', 'popularity_inv', 'popularity_tier', 'log_morning_odds', 'implied_prob']


## 3. ターゲット定義 & パイプラインラッパー

各予測ターゲットを `ModelingTarget` として定義し、`run_pipeline(target, df)` を呼ぶだけで  
モデル学習〜評価が完結する構造にする。

| ターゲット名 | タスク | ラベル | 用途 |
|---|---|---|---|
| `top3_cls` | binary_cls | finish_position ≤ 3 | 馬券圏内確率（Stage A） |
| `win_cls` | binary_cls | finish_position = 1 | 単勝確率（Stage A variant） |
| `finish_rank` | rank | finish_position | 着順ランキング（Stage B） |

In [10]:
from dataclasses import dataclass, field

# カテゴリ列・ID列・リーク列 — 特徴量から除外
_META_COLS: frozenset = frozenset([
    "race_id", "horse_id", "horse_number",
    "horse_name", "jockey_name", "trainer_name",
    "jockey_id", "trainer_id",
    "sire", "dam_sire", "sex_age",
    "venue", "venue_code", "surface", "direction",
    "grade", "race_class", "weather", "track_condition",
    "start_time", "race_name", "weight_rule", "course_type",
    # jt_stats メタ列
    "jt_result_date", "jt_race_datetime",
    "jt_row_jockey_id", "jt_row_trainer_id",
])

_LEAKAGE_COLS: frozenset = frozenset([
    "finish_position", "rank", "finish_pos",
    "win_odds", "popularity",
    "finish_time_sec", "last_3f_sec",
])


@dataclass
class ModelingTarget:
    name: str
    description: str
    label_col: str                       # 生ラベル列名
    task_type: str                       # "binary_cls" | "rank"
    place_threshold: int = 3             # binary_cls: finish_pos <= N なら正例
    extra_exclude: tuple = ()            # 追加で除外する列
    n_splits: int = 5
    n_seeds: int = 3                     # binary_cls のみ使用（多シード平均）


TARGETS: dict[str, ModelingTarget] = {
    "top3_cls": ModelingTarget(
        name="top3_cls",
        description="3着以内分類（馬券圏内確率）— Stage A",
        label_col="finish_position",
        task_type="binary_cls",
        place_threshold=3,
    ),
    "win_cls": ModelingTarget(
        name="win_cls",
        description="1着勝利分類（単勝確率）— Stage A variant",
        label_col="finish_position",
        task_type="binary_cls",
        place_threshold=1,
    ),
    "finish_rank": ModelingTarget(
        name="finish_rank",
        description="着順ランキング（LambdaRank）— Stage B",
        label_col="finish_position",
        task_type="rank",
        place_threshold=3,
    ),
}

print("ModelingTarget 定義完了:")
for t in TARGETS.values():
    print(f"  {t.name:15s} | {t.task_type:10s} | threshold={t.place_threshold} | {t.description}")

ModelingTarget 定義完了:
  top3_cls        | binary_cls | threshold=3 | 3着以内分類（馬券圏内確率）— Stage A
  win_cls         | binary_cls | threshold=1 | 1着勝利分類（単勝確率）— Stage A variant
  finish_rank     | rank       | threshold=3 | 着順ランキング（LambdaRank）— Stage B


In [11]:
def prepare_features(
    df: pd.DataFrame,
    target: ModelingTarget,
    date_col: str,
) -> tuple:
    """
    ターゲットに応じて学習用 df / X / y を準備する。

    Returns:
        df_train  : ラベルが有効な行のみ
        X         : 数値特徴量 DataFrame
        y         : ラベル Series
        feat_cols : X の列名リスト
    """
    exclude = (
        _META_COLS
        | _LEAKAGE_COLS
        | {date_col, target.label_col}
        | set(target.extra_exclude)
        | {c for c in df.columns if c.startswith("target_") or c.startswith("oof_")}
    )

    df_train = df[df[target.label_col].notna()].copy()
    df_train[target.label_col] = pd.to_numeric(df_train[target.label_col], errors="coerce")
    df_train = df_train[df_train[target.label_col].notna()].reset_index(drop=True)

    if target.task_type == "binary_cls":
        y = (df_train[target.label_col] <= target.place_threshold).astype(int)
    else:
        y = df_train[target.label_col].astype(float)

    feat_cols = [c for c in df_train.columns if c not in exclude]
    X = df_train[feat_cols].select_dtypes(include=[np.number])
    feat_cols = X.columns.tolist()

    return df_train, X, y, feat_cols


def run_pipeline(
    target: ModelingTarget,
    df_fe: pd.DataFrame,
    date_col: str = "race_date",
) -> dict:
    """
    ターゲット別にフルパイプライン（学習 → OOF評価 → 的中率計算）を実行する。

    - binary_cls → Stage A: LGBM + XGB + CatBoost
      出力スコア: 高いほど正例（馬券圏内）確率が高い
    - rank       → Stage B: LambdaRank
      出力スコア: 高いほど上位（1着側）を予測
    """
    print(f"\n{'='*60}")
    print(f"[{target.name}] {target.description}")
    print(f"{'='*60}")

    df_train, X, y, feat_cols = prepare_features(df_fe, target, date_col)
    print(f"学習データ: {df_train.shape} / 特徴量数: {len(feat_cols)}")
    if target.task_type == "binary_cls":
        print(f"正例率（≤{target.place_threshold}着）: {y.mean():.3f}")
    else:
        print(f"ラベル範囲: {y.min():.0f} 〜 {y.max():.0f}")

    groups_date = df_train[date_col]
    oof_preds = np.full(len(df_train), np.nan)
    models = []
    metrics: dict = {}
    min_rows = 500

    if target.task_type == "binary_cls":
        if len(df_train) >= min_rows:
            oof_preds, models = train_stage_a(
                X, y, groups_date,
                n_splits=target.n_splits,
                n_seeds=target.n_seeds,
            )
            valid = ~np.isnan(oof_preds) & (oof_preds > 0)
            if valid.sum() >= 50:
                from sklearn.metrics import roc_auc_score as _auc
                metrics["auc"] = _auc(y[valid], oof_preds[valid])
        else:
            print(f"  データ不足({len(df_train)} < {min_rows})のためスキップ")
            oof_preds[:] = y.mean()

    elif target.task_type == "rank":
        if len(df_train) >= min_rows and "race_id" in df_train.columns:
            oof_preds = train_stage_b(
                X, y, df_train["race_id"], groups_date,
                n_splits=target.n_splits,
            )
            valid = oof_preds != 0
            if valid.sum() >= 50:
                from scipy.stats import spearmanr as _spearman
                # oof_preds: 高いほど上位予測 → y(着順)と逆相関が正しい
                corr, _ = _spearman(y[valid], -oof_preds[valid])
                metrics["spearman"] = corr
        else:
            print(f"  データ不足 or race_id なし → スキップ")
            oof_preds[:] = 0.0

    # ── top-N 的中率（nlargest でスコアが高い馬を選ぶ → どちらも高=良で統一）──
    df_result = df_train.copy()
    df_result[f"oof_{target.name}"] = oof_preds
    score_col = f"oof_{target.name}"
    if "race_id" in df_result.columns:
        prec = top3_precision_per_race(df_result, score_col, target.label_col, top_n=target.place_threshold)
        metrics[f"top{target.place_threshold}_precision"] = prec
        print(f"  → 上位{target.place_threshold}頭的中率: {prec:.4f}")

    print(f"\n  metrics: {metrics}")
    return {
        "target": target.name,
        "df_result": df_result,
        "oof_preds": oof_preds,
        "feature_cols": feat_cols,
        "models": models,
        "metrics": metrics,
    }


print("prepare_features() / run_pipeline() 定義完了")

prepare_features() / run_pipeline() 定義完了


## 4. Purged GroupKFold（時系列バリデーション）

> IEEE Fraud: "train on months 1-4, skip month 5, predict month 6 — to mimic train/test gap"  
> Jane Street: "Purged Group Time-Series Split"  
> 共通原則: **「CV を信じろ。PublicLB は信じるな」**

In [12]:
from sklearn.model_selection import GroupKFold


class PurgedGroupTimeSeriesSplit:
    """
    時系列データ用 Purged GroupKFold。
    
    - groups: race_date をエンコードした整数（古い日付 = 小さい値）
    - purge_gap: 訓練最終日〜検証開始日のギャップ（日数）。情報リーク防止。
    
    参考: IEEE Fraud 1位「45-day gap」、Optiver「5-day gap」
    """

    def __init__(self, n_splits: int = 5, purge_gap_days: int = 7):
        self.n_splits = n_splits
        self.purge_gap_days = purge_gap_days

    def split(self, X, y=None, groups=None):
        """
        groups: pd.Series of datetime (race_date)
        """
        if groups is None:
            raise ValueError("groups (race_date) が必要です")

        groups = pd.to_datetime(groups)
        unique_dates = sorted(groups.unique())
        n_dates = len(unique_dates)
        fold_size = n_dates // (self.n_splits + 1)

        for fold in range(self.n_splits):
            train_end_idx = fold_size * (fold + 1)
            val_start_idx = train_end_idx + 1
            val_end_idx = min(train_end_idx + fold_size, n_dates)

            if val_start_idx >= n_dates:
                break

            train_end_date = unique_dates[train_end_idx - 1]
            val_start_date = unique_dates[val_start_idx]

            actual_gap = (val_start_date - train_end_date).days
            if actual_gap < self.purge_gap_days:
                from pandas import Timedelta
                target_val_start = train_end_date + Timedelta(days=self.purge_gap_days)
                filtered = [d for d in unique_dates if d >= target_val_start]
                if not filtered:
                    continue
                val_start_date = filtered[0]

            val_end_date = unique_dates[min(val_end_idx - 1, n_dates - 1)]

            train_mask = groups <= train_end_date
            val_mask = (groups >= val_start_date) & (groups <= val_end_date)

            train_idx = np.where(train_mask)[0]
            val_idx = np.where(val_mask)[0]

            if len(train_idx) == 0 or len(val_idx) == 0:
                continue

            yield train_idx, val_idx


print("PurgedGroupTimeSeriesSplit 定義完了")
print("  - n_splits=5, purge_gap_days=7 がデフォルト")
print("  - run_pipeline() 内で各ターゲットに対して自動使用される")

PurgedGroupTimeSeriesSplit 定義完了
  - n_splits=5, purge_gap_days=7 がデフォルト
  - run_pipeline() 内で各ターゲットに対して自動使用される


## 5. Adversarial Validation（分布シフト検出）

> IEEE Fraud 1位・AMEX 11位: "Adversarial Validation — drop features with high train/test discrepancy"  
> ここでは直近 30日を「テスト」として扱い、乖離の大きい特徴量を特定する

In [13]:
import lightgbm as lgb
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold


def adversarial_validation(
    df: pd.DataFrame,
    feature_cols: list,
    date_col: str,
    recent_days: int = 90,
) -> pd.DataFrame:
    """
    直近 recent_days 日のデータを「テスト」として、
    それ以前との分布差を LightGBM で検出する。
    AUC > 0.6 の特徴量は分布シフトが疑われる。
    """
    max_date = df[date_col].max()
    threshold_date = max_date - pd.Timedelta(days=recent_days)

    num_cols = [c for c in feature_cols if c in df.columns and pd.api.types.is_numeric_dtype(df[c])]
    X_adv = df[num_cols].fillna(-999)
    label = (df[date_col] > threshold_date).astype(int)

    cv = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
    oof_preds = np.zeros(len(X_adv))

    for tr_idx, va_idx in cv.split(X_adv, label):
        model = lgb.LGBMClassifier(
            n_estimators=100, learning_rate=0.1,
            max_depth=4, random_state=42, verbose=-1
        )
        model.fit(X_adv.iloc[tr_idx], label.iloc[tr_idx])
        oof_preds[va_idx] = model.predict_proba(X_adv.iloc[va_idx])[:, 1]

    auc = roc_auc_score(label, oof_preds)
    print(f"Adversarial AUC: {auc:.4f}  (0.5=no shift, >0.6=要注意)")

    final_model = lgb.LGBMClassifier(
        n_estimators=100, learning_rate=0.1,
        max_depth=4, random_state=42, verbose=-1
    )
    final_model.fit(X_adv, label)
    imp = pd.DataFrame({
        "feature": num_cols,
        "importance": final_model.feature_importances_,
    }).sort_values("importance", ascending=False)
    return imp


print("adversarial_validation() 定義完了")
print("  - run_pipeline() 呼び出し後に adversarial_validation(result['df_result'], ...) で利用可")

adversarial_validation() 定義完了
  - run_pipeline() 呼び出し後に adversarial_validation(result['df_result'], ...) で利用可


## 6. Stage A: 3着以内分類モデル

> Equity HCT: "Component A — Predicts P(event=1)"  
> Child Mind: "Repeated Stratified KFold + 多シード" → ノイズに対してロバスト

### LGBM + XGB + CatBoost 3モデルアンサンブル

In [14]:
import xgboost as xgb
from catboost import CatBoostClassifier

N_SEEDS = 3  # 多シード平均でノイズ安定化（Child Mind 金メダル戦略）

# LGBM params — DART boosting（AMEX 金メダル: "gold standard due to DART"）
LGBM_PARAMS_CLS = {
    "objective": "binary",
    "metric": "auc",
    "boosting_type": "dart",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "num_leaves": 63,
    "min_child_samples": 20,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbose": -1,
}

# XGBoost 2.0+ では early_stopping_rounds をコンストラクタに渡す
XGB_PARAMS_CLS = {
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "n_estimators": 500,
    "learning_rate": 0.05,
    "max_depth": 6,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "reg_alpha": 0.1,
    "reg_lambda": 1.0,
    "verbosity": 0,
    "early_stopping_rounds": 50,
}

CAT_PARAMS_CLS = {
    "iterations": 500,
    "learning_rate": 0.05,
    "depth": 6,
    "loss_function": "Logloss",
    "eval_metric": "AUC",
    "random_seed": 42,
    "verbose": 0,
}

print(f"モデルパラメータ設定完了 (N_SEEDS={N_SEEDS})")

モデルパラメータ設定完了 (N_SEEDS=3)


In [15]:
def train_stage_a(
    X: pd.DataFrame,
    y: pd.Series,
    groups_date: pd.Series,
    n_splits: int = 5,
    n_seeds: int = 3,
) -> tuple:
    """
    Stage A（3着以内分類）を Purged GroupKFold × 多シードで学習。

    Returns:
        oof_preds    : OOF 予測（Shape: len(X)）
        trained_models: (name, seed, fold, model) タプルのリスト
    """
    oof_preds = np.zeros(len(X))
    oof_counts = np.zeros(len(X))
    trained_models = []
    aucs = []

    X_arr = X.values.astype(np.float32)

    for seed in range(n_seeds):
        splitter = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge_gap_days=7)

        for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_arr, groups=groups_date)):
            X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
            y_tr, y_va = y.iloc[tr_idx].values, y.iloc[va_idx].values

            fold_preds = np.zeros(len(va_idx))
            n_models = 0

            # --- LGBM (DART: early_stopping は未対応のため省略) ---
            try:
                params = {**LGBM_PARAMS_CLS, "random_state": seed * 100 + fold_idx}
                lgbm_model = lgb.LGBMClassifier(**params)
                lgbm_model.fit(X_tr, y_tr)
                fold_preds += lgbm_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("lgbm", seed, fold_idx, lgbm_model))
            except Exception as e:
                print(f"  LGBM failed seed={seed} fold={fold_idx}: {e}")

            # --- XGBoost (early_stopping_rounds はコンストラクタに設定済み) ---
            try:
                params = {**XGB_PARAMS_CLS, "random_state": seed * 100 + fold_idx}
                xgb_model = xgb.XGBClassifier(**params)
                xgb_model.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
                fold_preds += xgb_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("xgb", seed, fold_idx, xgb_model))
            except Exception as e:
                print(f"  XGB failed seed={seed} fold={fold_idx}: {e}")

            # --- CatBoost ---
            try:
                params = {**CAT_PARAMS_CLS, "random_seed": seed * 100 + fold_idx}
                cat_model = CatBoostClassifier(**params)
                cat_model.fit(
                    X_tr, y_tr,
                    eval_set=(X_va, y_va),
                    early_stopping_rounds=50,
                    verbose=False,
                )
                fold_preds += cat_model.predict_proba(X_va)[:, 1]
                n_models += 1
                trained_models.append(("cat", seed, fold_idx, cat_model))
            except Exception as e:
                print(f"  CatBoost failed seed={seed} fold={fold_idx}: {e}")

            if n_models > 0:
                fold_preds /= n_models
                oof_preds[va_idx] += fold_preds
                oof_counts[va_idx] += 1

                fold_auc = roc_auc_score(y_va, fold_preds)
                aucs.append(fold_auc)
                print(f"  Seed {seed} Fold {fold_idx+1}: AUC={fold_auc:.4f} (n_models={n_models})")

    mask = oof_counts > 0
    oof_preds[mask] /= oof_counts[mask]

    if mask.sum() > 0:
        overall_auc = roc_auc_score(y[mask], oof_preds[mask])
        print(f"\n=== Stage A OOF AUC: {overall_auc:.4f} (fold mean: {np.mean(aucs):.4f} ± {np.std(aucs):.4f}) ===")

    return oof_preds, trained_models


print("train_stage_a() 定義完了")

train_stage_a() 定義完了


In [16]:
print("Stage A は run_pipeline() から呼ばれる — Section 10 の実行セルを参照")

Stage A は run_pipeline() から呼ばれる — Section 10 の実行セルを参照


## 7. Stage B: ランキングモデル（LambdaRank）

> Equity HCT: "Component B — Predicts ranking of event times"  
> OTTO Recommender: "LGBMRanker with LambdaRank loss"

着順ランキング学習。`group` パラメータ = 各レースの出走頭数。

In [17]:
from scipy.stats import spearmanr


def train_stage_b(
    X: pd.DataFrame,
    y: pd.Series,
    race_ids: pd.Series,
    groups_date: pd.Series,
    n_splits: int = 5,
) -> np.ndarray:
    """
    Stage B（着順ランキング）を LambdaRank で学習。

    Returns:
        oof_scores: OOF ランキングスコア（高いほど上位予測）
    """
    oof_scores = np.zeros(len(X))
    oof_counts = np.zeros(len(X))
    spearman_list = []

    X_arr = X.values.astype(np.float32)

    splitter = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge_gap_days=7)

    for fold_idx, (tr_idx, va_idx) in enumerate(splitter.split(X_arr, groups=groups_date)):
        X_tr, X_va = X_arr[tr_idx], X_arr[va_idx]
        y_tr = y.iloc[tr_idx].values
        y_va = y.iloc[va_idx].values
        race_tr = race_ids.iloc[tr_idx]
        race_va = race_ids.iloc[va_idx]

        tr_df = pd.DataFrame({"race_id": race_tr.values, "y": y_tr})
        group_sizes_tr = tr_df.groupby("race_id", sort=False).size().values

        # LambdaRank はラベルが非負整数を要求 → 着順を反転（1着 = 最高スコア）
        # max_pos - rank で 1着 = max_pos-1, 最下位 = 0
        max_pos = int(y_tr.max())
        y_tr_rank = (max_pos - y_tr).astype(int)
        y_tr_rank = np.clip(y_tr_rank, 0, None)

        ranker = lgb.LGBMRanker(
            objective="lambdarank",
            metric="ndcg",
            n_estimators=300,
            learning_rate=0.05,
            max_depth=6,
            num_leaves=63,
            min_child_samples=10,
            subsample=0.8,
            colsample_bytree=0.8,
            verbose=-1,
            random_state=42 + fold_idx,
        )
        try:
            ranker.fit(X_tr, y_tr_rank, group=group_sizes_tr)
            fold_scores = ranker.predict(X_va)
            oof_scores[va_idx] += fold_scores
            oof_counts[va_idx] += 1

            # スコアが高い = 上位予測 → 着順と逆相関が正しい
            corr, _ = spearmanr(y_va, -fold_scores)
            spearman_list.append(corr)
            print(f"  Fold {fold_idx+1}: Spearman={corr:.4f}")
        except Exception as e:
            print(f"  Fold {fold_idx+1}: LambdaRank failed: {e}")

    mask = oof_counts > 0
    oof_scores[mask] /= oof_counts[mask]

    if spearman_list:
        print(f"\n=== Stage B OOF Spearman: {np.mean(spearman_list):.4f} ± {np.std(spearman_list):.4f} ===")

    return oof_scores


print("train_stage_b() 定義完了")

train_stage_b() 定義完了


In [18]:
print("Stage B は run_pipeline() から呼ばれる — Section 10 の実行セルを参照")

Stage B は run_pipeline() から呼ばれる — Section 10 の実行セルを参照


## 8. 2段階スコア統合（Post-processing）

> Equity HCT 1位: `res = (1 - y_fun) * x_fun + y_fun`（Power Law Merge）  
> 競馬適用: Stage A（馬券圏内確率）× Stage B（ランキングスコア）を組み合わせる

### レース内整合後処理

> Optiver: "zero-sum post-processing — predictions should sum to zero"  
> IEEE Fraud: "UID Averaging — replace individual predictions with UID-level mean"

→ 競馬: 同レース内で予測スコアを相対スケールに正規化する

In [19]:
def merge_stages(
    df: pd.DataFrame,
    oof_a: np.ndarray,
    oof_b: np.ndarray,
    alpha: float = 0.5,
) -> np.ndarray:
    """
    Stage A（3着内確率）と Stage B（ランキングスコア）を統合。

    Equity HCT 1位の Power Law Merge を参考に:
      final = alpha * norm(stage_a) + (1-alpha) * norm(stage_b)

    alpha=0.5 を基準とし、Nelder-Mead で最適化する。
    """
    from sklearn.preprocessing import MinMaxScaler

    def race_normalize(scores: np.ndarray, race_ids: pd.Series) -> np.ndarray:
        normalized = np.zeros_like(scores, dtype=float)
        for race_id in race_ids.unique():
            mask = race_ids == race_id
            s = scores[mask]
            s_range = s.max() - s.min()
            normalized[mask] = (s - s.min()) / s_range if s_range > 0 else 0.5
        return normalized

    if "race_id" in df.columns:
        norm_a = race_normalize(oof_a, df["race_id"])
        norm_b = race_normalize(oof_b, df["race_id"])
    else:
        scaler = MinMaxScaler()
        norm_a = scaler.fit_transform(oof_a.reshape(-1, 1)).ravel()
        norm_b = scaler.fit_transform(oof_b.reshape(-1, 1)).ravel()

    return alpha * norm_a + (1 - alpha) * norm_b


print("merge_stages() 定義完了")
print("  - top3_cls と finish_rank の結果を結合する場合に使用")

merge_stages() 定義完了
  - top3_cls と finish_rank の結果を結合する場合に使用


## 9. alpha 最適化（Nelder-Mead）

> Child Mind: "Optimized Thresholds using Nelder-Mead"  
> Equity HCT 5位: "Greedy Ensemble Selection"  

Stage A と Stage B の混合比 alpha をレース内的中率（3着以内予測精度）で最適化する。

In [20]:
from scipy.optimize import minimize


def top3_precision_per_race(
    df: pd.DataFrame,
    score_col: str,
    true_col: str,
    top_n: int = 3,
) -> float:
    """各レースで上位 top_n 頭を予測し、実際に top_n 着以内だった割合を返す。"""
    if "race_id" not in df.columns:
        return 0.0
    precisions = []
    for _, grp in df.groupby("race_id"):
        if len(grp) < top_n:
            continue
        top_predicted = grp.nlargest(top_n, score_col).index
        actual_top = grp[grp[true_col] <= top_n].index
        precisions.append(len(set(top_predicted) & set(actual_top)) / top_n)
    return float(np.mean(precisions)) if precisions else 0.0


def optimize_alpha(
    df_shared: pd.DataFrame,
    oof_a: np.ndarray,
    oof_b: np.ndarray,
    label_col: str,
    top_n: int = 3,
) -> float:
    """
    merge_stages の alpha を Nelder-Mead で最適化して返す。
    df_shared は oof_a/oof_b と同じ行順の DataFrame（race_id 列が必要）。
    """
    def _neg_prec(alpha_arr):
        alpha = float(np.clip(alpha_arr[0], 0, 1))
        score = merge_stages(df_shared, oof_a, oof_b, alpha=alpha)
        tmp = df_shared.copy()
        tmp["_score"] = score
        return -top3_precision_per_race(tmp, "_score", label_col, top_n=top_n)

    res = minimize(_neg_prec, x0=[0.5], method="Nelder-Mead",
                   options={"xatol": 0.01, "fatol": 0.001, "maxiter": 50})
    best_alpha = float(np.clip(res.x[0], 0, 1))
    best_prec = -res.fun
    print(f"最適 alpha: {best_alpha:.3f}  →  上位{top_n}頭的中率: {best_prec:.4f}")
    return best_alpha


print("top3_precision_per_race() / optimize_alpha() 定義完了")

top3_precision_per_race() / optimize_alpha() 定義完了


## 10. ターゲット別パイプライン実行

`run_pipeline(target, df_fe)` を各ターゲットに対して呼ぶ。  
結果は `results` 辞書に格納され、後段の評価・特徴量重要度でも再利用する。

In [21]:
results: dict[str, dict] = {}

for target in TARGETS.values():
    results[target.name] = run_pipeline(target, df_fe, date_col=DATE_COL)


# ── 評価サマリー ──
print("\n" + "=" * 60)
print("Gold Medal Baseline — ターゲット別評価サマリー")
print("=" * 60)
for name, r in results.items():
    m = r["metrics"]
    parts = [f"  [{name}]"]
    if "auc" in m:
        parts.append(f"AUC={m['auc']:.4f}")
    if "spearman" in m:
        parts.append(f"Spearman={m['spearman']:.4f}")
    for k, v in m.items():
        if "precision" in k:
            parts.append(f"{k}={v:.4f}")
    print("  ".join(parts))


# ── Stage A + Stage B のアンサンブル（top3_cls × finish_rank）──
if "top3_cls" in results and "finish_rank" in results:
    print("\n" + "-" * 60)
    print("Stage A × Stage B アンサンブル最適化（Nelder-Mead alpha）")
    print("-" * 60)
    r_a = results["top3_cls"]
    r_b = results["finish_rank"]

    df_a = r_a["df_result"].set_index(["race_id", "horse_number"])
    df_b = r_b["df_result"].set_index(["race_id", "horse_number"])
    common_idx = df_a.index.intersection(df_b.index)

    if len(common_idx) >= 500:
        df_shared = df_a.loc[common_idx, ["finish_position"]].copy()
        oof_a_shared = df_a.loc[common_idx, "oof_top3_cls"].values
        # finish_rank のスコアは高=上位なので符号反転不要
        oof_b_shared = df_b.loc[common_idx, "oof_finish_rank"].values

        best_alpha = optimize_alpha(
            df_shared.reset_index(), oof_a_shared, oof_b_shared,
            label_col="finish_position", top_n=3,
        )
        final_score = merge_stages(df_shared.reset_index(), oof_a_shared, oof_b_shared, alpha=best_alpha)
        df_shared = df_shared.reset_index()
        df_shared["final_score"] = final_score
        prec_final = top3_precision_per_race(df_shared, "final_score", "finish_position", top_n=3)
        print(f"  最終スコア（alpha={best_alpha:.3f}）→ 上位3頭的中率: {prec_final:.4f}")
    else:
        print(f"  共通データ不足（{len(common_idx)}件）— スキップ")


print("\n" + "=" * 60)
print("金メダル戦略 実装確認")
print("=" * 60)
print(f"""
✓ make_mock_df()       全データソース対応モック（{df_raw.shape[1]}列）
✓ ModelingTarget       ターゲット別設定（top3_cls / win_cls / finish_rank）
✓ run_pipeline()       ターゲット別フルパイプラインラッパー
✓ LGBM(DART)+XGB+CatBoost  Stage A アンサンブル（N_SEEDS={N_SEEDS}）
✓ LambdaRank           Stage B ランキング学習（非負ラベル変換済み）
✓ PurgedGroupKFold     7日ギャップ付き時系列バリデーション
✓ merge_stages()       Stage A×B Power Law Merge
✓ optimize_alpha()     Nelder-Mead alpha 最適化
""")


[top3_cls] 3着以内分類（馬券圏内確率）— Stage A
学習データ: (2999, 109) / 特徴量数: 83
正例率（≤3着）: 0.250


  Seed 0 Fold 1: AUC=0.5257 (n_models=3)


  Seed 0 Fold 2: AUC=0.5603 (n_models=3)


  Seed 0 Fold 3: AUC=0.5672 (n_models=3)


  Seed 0 Fold 4: AUC=0.5302 (n_models=3)


...

  Fold 4: Spearman=0.1210


  Fold 5: Spearman=0.1394

=== Stage B OOF Spearman: 0.1247 ± 0.0300 ===


  → 上位3頭的中率: 0.2600

  metrics: {'spearman': np.float64(0.11324973689630119), 'top3_precision': 0.26}

Gold Medal Baseline — ターゲット別評価サマリー
  [top3_cls]  AUC=0.5365  top3_precision=0.2467
  [win_cls]  AUC=0.5040  top1_precision=0.0920
  [finish_rank]  Spearman=0.1132  top3_precision=0.2600

------------------------------------------------------------
Stage A × Stage B アンサンブル最適化（Nelder-Mead alpha）
------------------------------------------------------------


最適 alpha: 0.600  →  上位3頭的中率: 0.2587


  最終スコア（alpha=0.600）→ 上位3頭的中率: 0.2587

金メダル戦略 実装確認

✓ make_mock_df()       全データソース対応モック（84列）
✓ ModelingTarget       ターゲット別設定（top3_cls / win_cls / finish_rank）
✓ run_pipeline()       ターゲット別フルパイプラインラッパー
✓ LGBM(DART)+XGB+CatBoost  Stage A アンサンブル（N_SEEDS=3）
✓ LambdaRank           Stage B ランキング学習（非負ラベル変換済み）
✓ PurgedGroupKFold     7日ギャップ付き時系列バリデーション
✓ merge_stages()       Stage A×B Power Law Merge
✓ optimize_alpha()     Nelder-Mead alpha 最適化



## 11. 特徴量重要度（SHAP）

> AMEX 11位: "Null Importance for feature pruning"  
> Optiver 1位: "CatBoost feature importance to trim to 300 features"

In [22]:
# ターゲット別 LGBM 特徴量重要度（binary_cls ターゲットのみ）
for target_name, r in results.items():
    models_list = r.get("models", [])
    feat_cols = r.get("feature_cols", [])
    if not models_list or not feat_cols:
        continue

    lgbm_models = [m for name, seed, fold, m in models_list if name == "lgbm"]
    if not lgbm_models:
        continue

    imp_all = np.zeros(len(feat_cols))
    for m in lgbm_models:
        if len(m.feature_importances_) == len(feat_cols):
            imp_all += m.feature_importances_
    imp_all /= len(lgbm_models)

    imp_df = pd.DataFrame({
        "feature": feat_cols,
        "importance": imp_all,
    }).sort_values("importance", ascending=False)

    print(f"\n=== [{target_name}] 特徴量重要度 Top 20 ===")
    print(imp_df.head(20).to_string(index=False))

    n_zero = (imp_df["importance"] == 0).sum()
    print(f"\n重要度 0 の特徴量: {n_zero}/{len(imp_df)} 件")

if not any(r.get("models") for r in results.values()):
    print("学習済みモデルがありません（データ不足などでスキップされた可能性があります）")


=== [top3_cls] 特徴量重要度 Top 20 ===
                            feature  importance
         sire_x_track_bias_weighted    355.1333
     dam_sire_x_track_bias_weighted    332.4667
      track_bias_winner_3f_weighted    324.1333
             sire_x_track_bias_diff    307.4000
          track_bias_winner_3f_diff    293.8000
                  jockey_prior_top3    293.4667
                 idx_speed_recent_2    289.0000
                         field_size    286.7333
                             weight    268.8667
                jockey_cal90_starts    267.2667
      track_bias_winner_3f_prev_day    258.0000
         dam_sire_x_track_bias_diff    248.0667
                   idx_speed_course    241.2667
             jockey_prior_top3_rate    237.8667
              trainer_cal365_starts    237.8000
           trainer_roll5_avg_finish    235.0667
                 idx_speed_recent_3    234.4667
                 jockey_prior_races    234.2667
track_bias_winner_3f_same_day_prior    224.3333
      

## 17. 次のステップ（残タスク）

### 実装済み ✓

| 戦略 | セクション |
|---|---|
| LGBM(DART) + XGB + CatBoost アンサンブル | Section 6 |
| Purged GroupKFold（7日ギャップ） | Section 4 |
| Adversarial Validation | Section 5 |
| 2段階分解（Stage A: 分類 + Stage B: LambdaRank） | Section 6-7 |
| Nelder-Mead alpha 最適化 | Section 9 |
| **ターゲット別パイプラインラッパー** | Section 3 |
| **競馬固有特徴量（人気・季節性・血統適性）** | Section 2-4 |
| **Null Importance フィーチャープルーニング** | Section 13 |
| **Meta-feature スタッキング（L2）** | Section 14 |
| **Greedy Ensemble Selection** | Section 15 |
| **Optuna ハイパーパラメータ最適化** | Section 16 |

### 未実装（将来タスク）

| 戦略 | 内容 | 参考 |
|---|---|---|
| **TabM / TabPFN 追加** | 小データサブセットでの精度向上 | Equity HCT 1〜6位 |
| **Online Learning** | 直近レース結果で週次再学習（ドリフト対応） | Optiver, Enefit |
| **騎手・調教師統計のラグ** | 直近 N 週間の成績傾向（cumshift） | keiba 固有 |
| **Null Importance プルーニング後の再学習** | `kept_features` のみで run_pipeline 再実行 | AMEX 11位 |
| **MLflow 連携** | ハイパーパラメータ・メトリクスのトラッキング | keiba-vpn infra |

## 13. Null Importance フィーチャープルーニング

> AMEX 11位: "Null Importance for feature pruning — features important only for random labels are noise"

ランダムにシャッフルしたラベルで学習した「偽物の重要度」と比較し、  
実際の重要度 / null重要度 の比率が低い特徴量（ノイズ）を除去する。

In [23]:
def compute_null_importance(
    X: pd.DataFrame,
    y: pd.Series,
    n_rounds: int = 10,
    random_state: int = 42,
) -> pd.DataFrame:
    """
    Null Importance によるフィーチャープルーニング。

    Returns:
        pd.DataFrame: feature / real_imp / null_imp_mean / imp_ratio（降順）
    """
    rng_seed = np.random.default_rng(random_state)
    X_arr = X.values.astype(np.float32)
    null_imps = []

    for r in range(n_rounds):
        y_perm = y.sample(frac=1, random_state=int(rng_seed.integers(0, 9999))).reset_index(drop=True)
        model = lgb.LGBMClassifier(
            objective="binary", n_estimators=100, learning_rate=0.1,
            max_depth=5, num_leaves=31, verbose=-1,
            random_state=random_state + r,
        )
        model.fit(X_arr, y_perm)
        null_imps.append(model.feature_importances_)

    null_imp_mean = np.mean(null_imps, axis=0)
    null_imp_std = np.std(null_imps, axis=0)

    model_real = lgb.LGBMClassifier(
        objective="binary", n_estimators=100, learning_rate=0.1,
        max_depth=5, num_leaves=31, verbose=-1, random_state=random_state,
    )
    model_real.fit(X_arr, y)
    real_imp = model_real.feature_importances_

    result = pd.DataFrame({
        "feature": X.columns,
        "real_imp": real_imp,
        "null_imp_mean": null_imp_mean,
        "null_imp_std": null_imp_std,
        "imp_ratio": real_imp / (null_imp_mean + 1e-6),
    }).sort_values("imp_ratio", ascending=False).reset_index(drop=True)

    return result


# top3_cls ターゲットで Null Importance を計算
if "top3_cls" in results:
    r_top3 = results["top3_cls"]
    df_t, X_ni, y_ni, feat_ni = prepare_features(df_fe, TARGETS["top3_cls"], DATE_COL)
    print("Null Importance 計算中（10ラウンド）...")
    null_imp_df = compute_null_importance(X_ni, y_ni, n_rounds=10)

    print(f"\n特徴量数: {len(null_imp_df)}")
    print("\n[上位20: imp_ratio 高い = 意味のある特徴量]")
    print(null_imp_df.head(20)[["feature", "real_imp", "null_imp_mean", "imp_ratio"]].to_string(index=False))

    # プルーニング: imp_ratio が 1.5 以上のみ残す
    PRUNE_THRESHOLD = 1.5
    kept_features = null_imp_df[null_imp_df["imp_ratio"] >= PRUNE_THRESHOLD]["feature"].tolist()
    pruned_features = null_imp_df[null_imp_df["imp_ratio"] < PRUNE_THRESHOLD]["feature"].tolist()
    print(f"\n閾値 {PRUNE_THRESHOLD} → 残存: {len(kept_features)} 件 / 除去: {len(pruned_features)} 件")
    print("除去候補（先頭20）:", pruned_features[:20])
else:
    kept_features = []
    print("top3_cls の結果が未計算のため Null Importance をスキップ")

Null Importance 計算中（10ラウンド）...



特徴量数: 83

[上位20: imp_ratio 高い = 意味のある特徴量]
                      feature  real_imp  null_imp_mean  imp_ratio
                   field_size        35        11.5000     3.0435
          jockey_roll5_starts         6         2.3000     2.6087
            jockey_cal90_wins        31        15.0000     2.0667
        trainer_cal365_starts        47        25.7000     1.8288
            jockey_roll5_top3        10         5.5000     1.8182
             horse_prior_wins         7         4.0000     1.7500
          trainer_cal365_wins        41        27.7000     1.4801
       trainer_prior_win_rate        45        31.2000     1.4423
       jockey_prior_top3_rate        42        30.7000     1.3681
           idx_speed_recent_1        31        22.8000     1.3596
           jockey_roll10_wins        13         9.7000     1.3402
    finish_position_roll5mean        27        20.3000     1.3300
     finish_position_roll5std        34        26.1000     1.3027
       sire_x_track_bias_diff    

## 14. Meta-feature スタッキング（L2）

> AMEX 1位・Home Credit: "OOF predictions as meta-features for second-level model"

Stage 1 の3ターゲット（top3_cls / win_cls / finish_rank）の OOF をメタ特徴量として、  
L2 モデル（軽量 LGBM）で最終的な「3着以内確率」を再予測する 2段スタック。

In [24]:
def train_meta_stacking(
    results: dict,
    date_col: str,
    label_col: str = "finish_position",
    place_threshold: int = 3,
) -> dict:
    """
    Stage 1 の OOF をメタ特徴量として L2 LGBM モデルを学習する（2段スタック）。

    Returns:
        dict: df_meta, oof_meta, metrics
    """
    # 全ターゲットの OOF を race_id + horse_number で結合
    meta_dfs = []
    for name, r in results.items():
        df_r = r["df_result"][["race_id", "horse_number", f"oof_{name}"]].copy()
        meta_dfs.append(df_r)

    if not meta_dfs:
        print("meta-stacking に使える OOF がありません")
        return {}

    df_meta = meta_dfs[0]
    for df_r in meta_dfs[1:]:
        df_meta = df_meta.merge(df_r, on=["race_id", "horse_number"], how="inner")

    # ラベル付与（top3 バイナリ）
    df_label = results[list(results.keys())[0]]["df_result"][
        ["race_id", "horse_number", label_col, date_col]
    ].copy()
    df_meta = df_meta.merge(df_label, on=["race_id", "horse_number"], how="inner")

    oof_cols = [c for c in df_meta.columns if c.startswith("oof_")]
    y_meta = (df_meta[label_col] <= place_threshold).astype(int)
    X_meta = df_meta[oof_cols].fillna(0.0)
    groups = df_meta[date_col]

    print(f"メタ特徴量: {oof_cols}")
    print(f"学習データ: {X_meta.shape}  正例率: {y_meta.mean():.3f}")

    meta_oof = np.zeros(len(X_meta))
    meta_counts = np.zeros(len(X_meta))
    splitter = PurgedGroupTimeSeriesSplit(n_splits=5, purge_gap_days=7)

    for tr_idx, va_idx in splitter.split(X_meta.values, groups=groups):
        model = lgb.LGBMClassifier(
            objective="binary", n_estimators=300, learning_rate=0.05,
            max_depth=3, num_leaves=7, min_child_samples=10,
            subsample=0.8, colsample_bytree=0.8,
            verbose=-1, random_state=42,
        )
        model.fit(X_meta.iloc[tr_idx], y_meta.iloc[tr_idx])
        meta_oof[va_idx] += model.predict_proba(X_meta.iloc[va_idx])[:, 1]
        meta_counts[va_idx] += 1

    valid = meta_counts > 0
    meta_oof[valid] /= meta_counts[valid]
    df_meta["oof_meta_stack"] = meta_oof

    metrics = {}
    if valid.sum() >= 50:
        from sklearn.metrics import roc_auc_score as _auc
        auc = _auc(y_meta[valid], meta_oof[valid])
        prec = top3_precision_per_race(df_meta, "oof_meta_stack", label_col, top_n=place_threshold)
        metrics = {"auc": auc, f"top{place_threshold}_precision": prec}
        print(f"\nL2 Meta AUC: {auc:.4f}  |  上位{place_threshold}頭的中率: {prec:.4f}")

    return {"df_meta": df_meta, "oof_meta": meta_oof, "metrics": metrics}


# 実行
meta_result = train_meta_stacking(results, date_col=DATE_COL, label_col="finish_position")

# Stage 1 ベストと比較
if meta_result and "top3_cls" in results:
    baseline_prec = results["top3_cls"]["metrics"].get("top3_precision", 0)
    meta_prec = meta_result["metrics"].get("top3_precision", 0)
    print(f"\n比較: Stage1 top3_cls = {baseline_prec:.4f}  →  L2 Stack = {meta_prec:.4f}"
          f"  ({'↑改善' if meta_prec > baseline_prec else '↓低下（OOF不足の可能性）'})")

メタ特徴量: ['oof_top3_cls', 'oof_win_cls', 'oof_finish_rank']
学習データ: (2999, 3)  正例率: 0.250



L2 Meta AUC: 0.5112  |  上位3頭的中率: 0.2587

比較: Stage1 top3_cls = 0.2467  →  L2 Stack = 0.2587  (↑改善)


## 15. Greedy Ensemble Selection

> Equity HCT 5位: "Greedy Ensemble Selection (Caruana et al. 2004)"  
> Optiver: "Adding more models improves the ensemble only if they differ from the current blend"

各フォールドのモデル OOF を候補として、的中率を最大化するサブセットを貪欲法で選択する。  
単純な平均より多様なモデルの組み合わせを見つけられる。

In [25]:
def greedy_ensemble_selection(
    candidates: list,        # list of (name, oof_array)
    df_ref: pd.DataFrame,    # race_id + label_col を持つ DataFrame（OOFと同じ行順）
    label_col: str,
    top_n: int = 3,
    n_iter: int = 50,
) -> tuple:
    """
    Greedy Ensemble Selection (Caruana et al. 2004).

    各ステップで「追加すると的中率が最も上がる」候補を選択し、
    単純な平均で統合する。復元あり選択（同じモデルを複数回選択可能）。

    Returns:
        selected_names : 選択されたモデル名のリスト（重複あり）
        ensemble_oof   : 最終アンサンブル OOF スコア
        history        : ステップごとの的中率推移
    """
    n_samples = len(candidates[0][1])
    ensemble = np.zeros(n_samples)
    selected_names = []
    history = []

    for step in range(n_iter):
        best_score = -np.inf
        best_idx = -1

        for idx, (name, oof) in enumerate(candidates):
            n_current = len(selected_names)
            candidate_ensemble = (ensemble * n_current + oof) / (n_current + 1)
            df_tmp = df_ref.copy()
            df_tmp["_score"] = candidate_ensemble
            score = top3_precision_per_race(df_tmp, "_score", label_col, top_n=top_n)
            if score > best_score:
                best_score = score
                best_idx = idx

        if best_idx >= 0:
            name, oof = candidates[best_idx]
            n_current = len(selected_names)
            ensemble = (ensemble * n_current + oof) / (n_current + 1)
            selected_names.append(name)
            history.append(best_score)

    return selected_names, ensemble, history


# top3_cls の全フォールドモデル OOF を候補として Greedy Selection
if "top3_cls" in results:
    r_a = results["top3_cls"]
    df_ref_ge = r_a["df_result"][["race_id", "horse_number", "finish_position"]].reset_index(drop=True)

    # 各 (model_type, seed, fold) のフォールド OOF をそれぞれ再計算するのは重いため、
    # 代わりに各ターゲットの OOF を候補として Greedy Selection する
    ge_candidates = []
    for name, r in results.items():
        oof = r["oof_preds"]
        df_r = r["df_result"].set_index(["race_id", "horse_number"])
        df_ref_ge2 = df_ref_ge.set_index(["race_id", "horse_number"])
        common = df_ref_ge2.index.intersection(df_r.index)
        if len(common) > 500:
            aligned_oof = df_r.loc[common, f"oof_{name}"].values
            df_common = df_ref_ge2.loc[common].reset_index()
            ge_candidates.append((name, aligned_oof))

    if len(ge_candidates) >= 2:
        print(f"Greedy Ensemble 候補: {[n for n, _ in ge_candidates]}")
        df_common_final = results[list(results.keys())[0]]["df_result"][
            ["race_id", "horse_number", "finish_position"]
        ].copy()
        # 共通インデックスに絞る
        idx0 = results["top3_cls"]["df_result"].set_index(["race_id", "horse_number"]).index
        for name, r in results.items():
            idx0 = idx0.intersection(r["df_result"].set_index(["race_id", "horse_number"]).index)

        df_ge_ref = results["top3_cls"]["df_result"].set_index(["race_id", "horse_number"]).loc[idx0, ["finish_position"]].reset_index()
        aligned_candidates = []
        for name, r in results.items():
            df_r = r["df_result"].set_index(["race_id", "horse_number"])
            oof_aligned = df_r.loc[idx0, f"oof_{name}"].values
            aligned_candidates.append((name, oof_aligned))

        sel_names, ge_ensemble, ge_history = greedy_ensemble_selection(
            aligned_candidates, df_ge_ref, "finish_position", top_n=3, n_iter=30
        )

        df_ge_ref["oof_greedy"] = ge_ensemble
        final_prec = top3_precision_per_race(df_ge_ref, "oof_greedy", "finish_position", top_n=3)
        print(f"\n選択ステップ ({len(sel_names)}): {sel_names}")
        print(f"Greedy Ensemble 上位3頭的中率: {final_prec:.4f}")
        print(f"的中率推移（先頭10ステップ）: {[f'{v:.4f}' for v in ge_history[:10]]}")
    else:
        print("候補が2件未満のため Greedy Selection をスキップ")
else:
    print("top3_cls の結果が未計算のためスキップ")

Greedy Ensemble 候補: ['top3_cls', 'win_cls', 'finish_rank']



選択ステップ (30): ['finish_rank', 'win_cls', 'finish_rank', 'win_cls', 'finish_rank', 'win_cls', 'finish_rank', 'top3_cls', 'win_cls', 'win_cls', 'top3_cls', 'top3_cls', 'win_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'finish_rank', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'win_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls', 'top3_cls']
Greedy Ensemble 上位3頭的中率: 0.2667
的中率推移（先頭10ステップ）: ['0.2600', '0.2613', '0.2613', '0.2613', '0.2613', '0.2613', '0.2613', '0.2613', '0.2613', '0.2613']


## 16. Optuna ハイパーパラメータ最適化

> Mitsui 7位, ICR 6位: "Bayesian Hyperparameter Optimization with Optuna"

Purged GroupKFold CV を Optuna の目標関数として LGBM パラメータを最適化する。  
DART は Optuna と相性が悪い（木の削除でスコアが不安定になる）ため `gbdt` で探索し、  
最終学習時のみ `dart` に切り替える。

In [26]:
def tune_lgbm_optuna(
    X: pd.DataFrame,
    y: pd.Series,
    groups_date: pd.Series,
    n_trials: int = 30,
    n_splits: int = 3,
    random_state: int = 42,
) -> object:
    """
    Optuna で LGBM のハイパーパラメータを最適化する。

    Returns:
        optuna.Study オブジェクト
    """
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)

    X_arr = X.values.astype(np.float32)

    def objective(trial):
        params = {
            "boosting_type": "gbdt",   # DART は Optuna と相性悪いので探索は gbdt
            "objective": "binary",
            "metric": "auc",
            "n_estimators": trial.suggest_int("n_estimators", 100, 600),
            "learning_rate": trial.suggest_float("learning_rate", 0.01, 0.15, log=True),
            "max_depth": trial.suggest_int("max_depth", 3, 9),
            "num_leaves": trial.suggest_int("num_leaves", 15, 127),
            "min_child_samples": trial.suggest_int("min_child_samples", 10, 80),
            "subsample": trial.suggest_float("subsample", 0.5, 1.0),
            "colsample_bytree": trial.suggest_float("colsample_bytree", 0.5, 1.0),
            "reg_alpha": trial.suggest_float("reg_alpha", 1e-4, 10.0, log=True),
            "reg_lambda": trial.suggest_float("reg_lambda", 1e-4, 10.0, log=True),
            "verbose": -1,
            "random_state": random_state + trial.number,
        }

        splitter = PurgedGroupTimeSeriesSplit(n_splits=n_splits, purge_gap_days=7)
        fold_aucs = []
        for tr_idx, va_idx in splitter.split(X_arr, groups=groups_date):
            model = lgb.LGBMClassifier(**params)
            model.fit(X_arr[tr_idx], y.iloc[tr_idx])
            pred = model.predict_proba(X_arr[va_idx])[:, 1]
            fold_aucs.append(roc_auc_score(y.iloc[va_idx], pred))
        return float(np.mean(fold_aucs))

    study = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=random_state),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5),
    )
    study.optimize(objective, n_trials=n_trials, show_progress_bar=False)

    print(f"\n最良 AUC: {study.best_value:.4f}  (trial #{study.best_trial.number})")
    print("最良パラメータ:")
    for k, v in study.best_params.items():
        print(f"  {k}: {v}")

    return study


# top3_cls ターゲットで Optuna チューニング（30 trials）
OPTUNA_N_TRIALS = 30

if "top3_cls" in results:
    df_t_opt, X_opt, y_opt, feat_opt = prepare_features(df_fe, TARGETS["top3_cls"], DATE_COL)
    print(f"Optuna チューニング開始（{OPTUNA_N_TRIALS} trials, 3-fold CV）...")
    optuna_study = tune_lgbm_optuna(
        X_opt, y_opt, df_t_opt[DATE_COL],
        n_trials=OPTUNA_N_TRIALS, n_splits=3,
    )

    # 最良パラメータを LGBM_PARAMS_CLS に反映（dart に切り替え）
    best_params_opt = {
        **LGBM_PARAMS_CLS,
        **{k: v for k, v in optuna_study.best_params.items()},
        "boosting_type": "dart",   # 最終学習は dart
        "verbose": -1,
    }
    print(f"\n最終学習パラメータ（dart で適用）:")
    for k, v in best_params_opt.items():
        print(f"  {k}: {v}")

    # 最良パラメータで top3_cls を再学習
    print("\n最適化パラメータで top3_cls を再学習中...")
    LGBM_PARAMS_CLS_TUNED = best_params_opt
    # run_pipeline で使われるグローバル変数を更新
    _original_params = LGBM_PARAMS_CLS.copy()
    LGBM_PARAMS_CLS.update(best_params_opt)
    result_tuned = run_pipeline(TARGETS["top3_cls"], df_fe, date_col=DATE_COL)
    LGBM_PARAMS_CLS.clear()
    LGBM_PARAMS_CLS.update(_original_params)  # 元に戻す

    baseline_auc = results["top3_cls"]["metrics"].get("auc", 0)
    tuned_auc = result_tuned["metrics"].get("auc", 0)
    print(f"\nチューニング前 AUC: {baseline_auc:.4f}  →  後: {tuned_auc:.4f}")
else:
    optuna_study = None
    print("top3_cls の結果が未計算のため Optuna をスキップ")

Optuna チューニング開始（30 trials, 3-fold CV）...



最良 AUC: 0.5149  (trial #14)
最良パラメータ:
  n_estimators: 174
  learning_rate: 0.03090558312134731
  max_depth: 5
  num_leaves: 57
  min_child_samples: 12
  subsample: 0.8834854882792176
  colsample_bytree: 0.9161498340015772
  reg_alpha: 9.417536673616803
  reg_lambda: 0.012662488734587064

最終学習パラメータ（dart で適用）:
  objective: binary
  metric: auc
  boosting_type: dart
  n_estimators: 174
  learning_rate: 0.03090558312134731
  max_depth: 5
  num_leaves: 57
  min_child_samples: 12
  subsample: 0.8834854882792176
  colsample_bytree: 0.9161498340015772
  reg_alpha: 9.417536673616803
  reg_lambda: 0.012662488734587064
  verbose: -1

最適化パラメータで top3_cls を再学習中...

[top3_cls] 3着以内分類（馬券圏内確率）— Stage A
学習データ: (2999, 109) / 特徴量数: 83
正例率（≤3着）: 0.250


  Seed 0 Fold 1: AUC=0.5513 (n_models=3)


  Seed 0 Fold 2: AUC=0.5813 (n_models=3)


  Seed 0 Fold 3: AUC=0.5869 (n_models=3)


...

  Seed 2 Fold 2: AUC=0.5480 (n_models=3)


  Seed 2 Fold 3: AUC=0.5663 (n_models=3)


  Seed 2 Fold 4: AUC=0.5341 (n_models=3)


  Seed 2 Fold 5: AUC=0.5820 (n_models=3)

=== Stage A OOF AUC: 0.5555 (fold mean: 0.5560 ± 0.0157) ===


  → 上位3頭的中率: 0.2613

  metrics: {'auc': 0.5555450931157145, 'top3_precision': 0.2613333333333333}

チューニング前 AUC: 0.5365  →  後: 0.5555
